# Feature Extraction — преобразование признаков простыми словами

Учебный ноутбук по материалам плаката **FEATURE EXTRACTION** (`python.pdf`).

> **Важно:** в PDF тема — **извлечение / преобразование** признаков (PCA, LDA, NCA, t-SNE),  
> а не Feature **Selection** (отбор столбцов).  
> Selection *выбирает* из существующих; Extraction *строит новые* оси/координаты.

| Selection | Extraction |
|-----------|------------|
| оставить age, income | заменить 100 признаков на PC1…PC10 |
| SelectKBest, Lasso | PCA, LDA, NCA |
| имена признаков те же | новые: PC1, LD1… (часто без «человеческого» смысла) |

**Методы плаката:** PCA · LDA · NCA · t-SNE

Запускайте ячейки **сверху вниз**.

---

## План

1. [Словарь](#dict)
2. [PCA — главные компоненты](#pca)
3. [Зачем StandardScaler перед PCA](#scale)
4. [explained_variance_ratio_ и число компонент](#var)
5. [Утечка данных и Pipeline](#pipe)
6. [PCA + K-Fold / подбор n_components](#cv)
7. [Когда PCA нужен и когда нет](#when)
8. [LDA — линейный дискриминантный анализ](#lda)
9. [NCA — для «умного» KNN](#nca)
10. [t-SNE — только визуализация](#tsne)
11. [Сравнение методов](#cmp)
12. [Шпаргалка](#итог)


<a id="dict"></a>
## 1. Словарь

| Термин | Простыми словами |
|--------|------------------|
| **Feature Extraction** | Построить **новые** признаки (часто меньшей размерности) |
| **Feature Selection** | **Выбрать** подмножество старых признаков |
| **Компонента (PC)** | Новая ось = комбинация старых признаков |
| **Дисперсия** | «Разброс» данных; PCA любит направления с большим разбросом |
| **Проекция** | Координаты точек в новых осях |
| **`fit` / `transform`** | Выучить оси на train → применить к train/test |
| **С учителем / без** | Смотрит ли метод на **y** (метки) |

```text
Без учителя (unsupervised):  PCA, t-SNE   — только X
С учителем (supervised):     LDA, NCA     — X и y
```


<a id="pca"></a>
## 2. PCA (Principal Component Analysis)

### Идея одной фразой

> Найти новые оси, где данные **сильнее всего размазаны**,  
> и спроецировать точки на первые несколько осей — сохранив **максимум информации** (дисперсии).

Пример с плаката: было **100** признаков → **10** компонент ≈ **95%** информации.

### Под капотом (очень грубо)

1. Смотрит, как признаки **связаны** (ковариации).  
2. Находит направления (собственные векторы), где разброс **максимален**.  
3. Первая ось — PC1 (макс. дисперсия), вторая — PC2 (макс. среди **ортогональных** к PC1) и т.д.  
4. Проецирует данные: вместо $(x_1,\ldots,x_p)$ получаем $(z_1,\ldots,z_k)$, $k \ll p$.

### Применение

1. Уменьшение размерности  
2. Ускорение обучения  
3. Борьба с **мультиколлинеарностью** (компоненты ортогональны)  
4. **Визуализация** (PC1 vs PC2)

### Минус

Новые признаки — **PC1, PC2, PC3…**  
Их почти нельзя объяснить бизнесу как «возраст» или «доход».

### Критично

**PCA не использует y.**  
Он сохраняет разброс $X$, а не «то, что нужно для предсказания».  
Иногда качество модели **падает** — это нормально, метод не обязан помогать прогнозу.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score

# Игрушка: 2 коррелированных признака → 1 главная ось
rng = np.random.default_rng(0)
t = rng.normal(size=200)
X2 = np.column_stack([t + rng.normal(0, 0.15, 200),
                      2 * t + rng.normal(0, 0.15, 200)])

scaler = StandardScaler()
Xs = scaler.fit_transform(X2)
pca2 = PCA(n_components=2).fit(Xs)
Z = pca2.transform(Xs)

print("explained_variance_ratio_:", np.round(pca2.explained_variance_ratio_, 3))
print("сумма:", round(pca2.explained_variance_ratio_.sum(), 3))
print("→ почти вся информация в PC1 (признаки сильно коррелированы)")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(Xs[:, 0], Xs[:, 1], s=12, alpha=0.7)
# направления компонент
origin = np.zeros(2)
for i, c in enumerate(["red", "orange"]):
    v = pca2.components_[i] * 3 * np.sqrt(pca2.explained_variance_[i])
    ax[0].arrow(0, 0, v[0], v[1], head_width=0.15, color=c, length_includes_head=True,
                label=f"PC{i+1}")
ax[0].set_title("Исходные (scaled) + оси PCA")
ax[0].set_aspect("equal"); ax[0].legend(); ax[0].grid(True, alpha=0.3)

ax[1].scatter(Z[:, 0], Z[:, 1], s=12, alpha=0.7)
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")
ax[1].set_title("Координаты в пространстве PCA")
ax[1].set_aspect("equal"); ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


<a id="scale"></a>
## 3. Почему перед PCA почти всегда нужен StandardScaler?

Пример с плаката:

| Признак | Диапазон |
|---------|----------|
| age | 18–80 |
| salary | 30 000–500 000 |

Без масштаба **salary** задавит age: PCA решит, что «главное направление» — почти только зарплата,  
потому что её **дисперсия в абсолютных единицах** огромна.  
Хотя причина может быть просто в **рублях vs годах**.

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=10)
Z = pca.fit_transform(X_scaled)
```

**Вывод:** разные единицы измерения → **сначала scale, потом PCA**.


In [ ]:
# age vs salary: без scale PCA «смотрит» только на salary
age = rng.uniform(18, 80, 300)
salary = rng.uniform(30_000, 500_000, 300)
X_raw = np.column_stack([age, salary])

pca_raw = PCA(n_components=2).fit(X_raw)
pca_sc = PCA(n_components=2).fit(StandardScaler().fit_transform(X_raw))

print("Доли дисперсии БЕЗ scaler:", np.round(pca_raw.explained_variance_ratio_, 4))
print("  loadings PC1 (age, salary):", np.round(np.abs(pca_raw.components_[0]), 4))
print("Доли дисперсии СО scaler:  ", np.round(pca_sc.explained_variance_ratio_, 4))
print("  loadings PC1 (age, salary):", np.round(np.abs(pca_sc.components_[0]), 4))
print("→ без scale |loading| salary ≈ 1; age почти не участвует.")


<a id="var"></a>
## 4. `explained_variance_ratio_` и сколько компонент брать

После `fit` у PCA есть:

```python
pca.explained_variance_ratio_   # доля дисперсии каждой компоненты
np.cumsum(pca.explained_variance_ratio_)  # накопленная
```

Пример с плаката (условный):

| Компонента | Доля |
|------------|------|
| PC1 | 50% |
| PC2 | 25% |
| PC3 | 15% |
| PC4 | 7% |
| … | … |

Первые три ≈ **90%** — часто достаточно для сжатия.

### Два способа задать `n_components`

```python
PCA(n_components=10)     # ровно 10 осей
PCA(n_components=0.95)   # минимум компонент, чтобы покрыть ≥ 95% дисперсии
```

> **Уточнение:** «95% дисперсии» — хорошая **эвристика**, не гарантия лучшего accuracy.  
> Финальный выбор лучше подтверждать **CV по качеству модели**.


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("Исходная размерность:", X.shape)

Xs = StandardScaler().fit_transform(X)
pca_full = PCA().fit(Xs)
cum = np.cumsum(pca_full.explained_variance_ratio_)

print("Компонент для ≥95% дисперсии:", int(np.searchsorted(cum, 0.95) + 1))
print("для ≥99%:", int(np.searchsorted(cum, 0.99) + 1))

plt.figure(figsize=(7, 3.5))
plt.plot(np.arange(1, len(cum)+1), cum, marker="o", ms=3)
plt.axhline(0.95, color="red", ls="--", label="95%")
plt.xlabel("число компонент"); plt.ylabel("накопленная доля дисперсии")
plt.title("Scree / cumulative explained variance")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# sklearn сам подберёт k при float
Z95 = PCA(n_components=0.95).fit_transform(Xs)
print("Форма при n_components=0.95:", Z95.shape)


<a id="pipe"></a>
## 5. Утечка данных и Pipeline

### Нельзя

```python
pca.fit_transform(X)          # на всём датасете до split
# или
pca.fit(np.vstack([X_train, X_test]))
```

PCA «увидит» test: средние, дисперсии, направления осей — **leakage**.

### Правильно

```python
pca.fit(X_train)
Z_train = pca.transform(X_train)
Z_test  = pca.transform(X_test)
```

### Как в проде и в коде — Pipeline

```python
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("model", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
pipe.predict(X_test)  # scaler и pca только transform, без переобучения
```

На `predict` для новых данных:

1. `StandardScaler.transform` (те же mean/std, что с train)  
2. `PCA.transform` (те же оси)  
3. `model.predict`


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
    ("model", LogisticRegression(max_iter=2000)),
])
pipe.fit(X_train, y_train)
acc = accuracy_score(y_test, pipe.predict(X_test))
n_comp = pipe.named_steps["pca"].n_components_
print(f"Test accuracy: {acc:.3f}")
print(f"PCA выбрал компонент: {n_comp} из {X.shape[1]}")
print("Pipeline гарантирует: fit scaler/pca только на train.")


<a id="cv"></a>
## 6. PCA внутри K-Fold и подбор числа компонент

Как на плакате: при `cross_val_score(pipe, X, y, cv=5)` **в каждом фолде**:

1. Scaler.fit(train_fold)  
2. PCA.fit(train_fold)  
3. Model.fit  
4. evaluate на valid_fold  

PCA **заново** для каждого фолда — это правильно.

### Подбор `n_components` через GridSearchCV

```python
param_grid = {"pca__n_components": [5, 10, 15, 20, 0.9, 0.95]}
GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy")
```

Сравнивают **качество модели**, а не только % дисперсии.


In [ ]:
pipe_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("model", LogisticRegression(max_iter=2000)),
])
grid = GridSearchCV(
    pipe_cv,
    param_grid={"pca__n_components": [2, 5, 10, 15, 20, 0.95]},
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid.fit(X, y)
print("Лучшие params:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 3))

# baseline без PCA
base = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000)),
])
print("CV accuracy без PCA:", round(cross_val_score(base, X, y, cv=5).mean(), 3))


<a id="when"></a>
## 7. Когда PCA стоит применять / не стоит

### Стоит (по плакату)

1. Признаков **много**  
2. Между ними **сильные корреляции**  
3. Модель **долго** учится  
4. Модели, чувствительные к размерности/корреляции: **линейные, SVM, KNN**  
5. Нужна **2D/3D** картинка данных  

Особенно: 500 признаков → 40 компонент, сохраняющих ~95% дисперсии.

### Лучше не надо

1. Всего 10–20 **понятных** признаков (income, age, debt…)  
2. Нужна **интерпретируемость** коэффициентов  
3. **Деревья / Random Forest / бустинги** — сами терпят много признаков и нелинейности; PCA часто не нужен  

### Модели (ориентир)

| Модель | PCA |
|--------|-----|
| Linear / Logistic / SVM / KNN | часто **полезен** |
| Decision Tree / RF / XGB / LGBM / CatBoost | обычно **не обязателен** |


<a id="lda"></a>
## 8. LDA (Linear Discriminant Analysis)

### Чем отличается от PCA

| | PCA | LDA |
|--|-----|-----|
| Использует **y**? | **Нет** | **Да** |
| Цель осей | max **дисперсия** данных | max **разделение классов** |
| Задача | сжатие / визуализация / предобработка | сжатие **для классификации** + сам классификатор |

Аналогия с плаката: «кошки и собаки».  
PCA сохранит «самый пёстрый» разброс (может быть бесполезен для классов).  
LDA ищет ось, где **кошки и собаки максимально далеко**.

### Идея (3 шага, упрощённо)

1. **Межклассовая** дисперсия $S_b$ — насколько центры классов далеки.  
2. **Внутриклассовая** $S_w$ — насколько точки размазаны **внутри** класса.  
3. Найти проекцию, где $S_b$ велика, а $S_w$ мала (классы кучные и далёкие друг от друга).

### Важно

- LDA — метод **с учителем**: нужен `y`.  
- Число компонент LDA ≤ **число классов − 1** (для 2 классов → максимум 1 ось).  
- Перед LDA часто тоже **масштабируют** признаки.  
- `fit` только на train.

```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = LinearDiscriminantAnalysis(n_components=1)
Z = lda.fit_transform(X_train, y_train)
```


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Два «облака» — PCA vs LDA
Xg, yg = make_classification(
    n_samples=400, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.2, random_state=4
)
Xg = StandardScaler().fit_transform(Xg)

Z_pca = PCA(n_components=2).fit_transform(Xg)
Z_lda = LinearDiscriminantAnalysis(n_components=1).fit_transform(Xg, yg)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].scatter(Xg[:, 0], Xg[:, 1], c=yg, cmap="coolwarm", s=12)
axes[0].set_title("Исходные 2D")
axes[1].scatter(Z_pca[:, 0], Z_pca[:, 1], c=yg, cmap="coolwarm", s=12)
axes[1].set_title("PCA (max дисперсия)")
axes[2].scatter(Z_lda[:, 0], np.zeros_like(Z_lda[:, 0]), c=yg, cmap="coolwarm", s=12)
axes[2].set_title("LDA 1D (max разделение классов)")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# LDA как классификатор + как extractor в Pipeline
pipe_lda = Pipeline([
    ("scaler", StandardScaler()),
    ("lda", LinearDiscriminantAnalysis()),
    # LDA сам может быть финальным классификатором:
])
# проще: LDA.fit predict
lda_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearDiscriminantAnalysis()),
])
print("CV accuracy LDA-classifier:",
      round(cross_val_score(lda_clf, X, y, cv=5).mean(), 3))


<a id="nca"></a>
## 9. NCA (Neighbourhood Components Analysis)

### Идея

> «Преобразовать пространство так, чтобы объекты **одного класса** стали **ближе**,  
> а разных — дальше» — особенно чтобы **KNN** лучше работал.

- Использует **y** (с учителем).  
- Учит линейное преобразование (метрика в духе Махаланобиса).  
- На практике встречается **реже**, чем PCA/LDA.  
- Имеет смысл, если дальше **KNN** (или другой distance-based метод).

```python
from sklearn.neighbors import NeighborhoodComponentsAnalysis, KNeighborsClassifier

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("nca", NeighborhoodComponentsAnalysis(n_components=2, random_state=0)),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
```

**Уточнение:** NCA дороже PCA; на больших данных может быть медленным.  
Не путать с t-SNE: NCA **для моделей**, t-SNE — **для картинок**.


In [ ]:
from sklearn.neighbors import NeighborhoodComponentsAnalysis, KNeighborsClassifier

# Сравнение KNN vs NCA+KNN на cancer (сжатие до 2D для наглядности)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

knn_only = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(5)),
])
nca_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("nca", NeighborhoodComponentsAnalysis(n_components=5, random_state=0, max_iter=100)),
    ("knn", KNeighborsClassifier(5)),
])

print("KNN CV:", round(cross_val_score(knn_only, X, y, cv=3).mean(), 3))
print("NCA+KNN CV:", round(cross_val_score(nca_knn, X, y, cv=3).mean(), 3))
print("(На разных датасетах NCA может дать +/−; это не серебряная пуля.)")


<a id="tsne"></a>
## 10. t-SNE — визуализация, не «фичи для модели»

### Идея

В высокой размерности точки-соседи → в 2D/3D они тоже должны быть **рядом**.  
Получаются красивые «кучки» для статей и EDA.

### Где применять

- Посмотреть структуру данных, кластеры, выбросы.  
- **Не** как стандартный шаг `Pipeline` перед LogisticRegression.

### Почему не для обучения (важные уточнения)

1. Цель — **картинка**, не сжатие под метрику модели.  
2. В sklearn у `TSNE` обычно делают `fit_transform` на **том наборе, который рисуете**;  
   классического стабильного `transform` для «новых клиентов как в PCA» **нет** (в отличие от PCA/LDA).  
3. Результат **стохастический** (random_state), зависит от perplexity.  
4. Оси t-SNE **не** интерпретировать как PC1/PC2 с % дисперсии.

```python
from sklearn.manifold import TSNE
Z = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X_scaled)
plt.scatter(Z[:,0], Z[:,1], c=y, s=10)
```


In [ ]:
from sklearn.manifold import TSNE

# t-SNE на breast cancer (подвыборка для скорости)
idx = rng.choice(len(X), size=400, replace=False)
Xs_full = StandardScaler().fit_transform(X)
Z_tsne = TSNE(n_components=2, perplexity=30, init="pca",
              learning_rate="auto", random_state=0).fit_transform(Xs_full[idx])
Z_pca_viz = PCA(n_components=2).fit_transform(Xs_full[idx])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(Z_pca_viz[:, 0], Z_pca_viz[:, 1], c=y[idx], cmap="coolwarm", s=12)
ax[0].set_title("PCA 2D (линейная проекция)")
ax[1].scatter(Z_tsne[:, 0], Z_tsne[:, 1], c=y[idx], cmap="coolwarm", s=12)
ax[1].set_title("t-SNE 2D (соседство, для глаз)")
for a in ax:
    a.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("t-SNE: смотрим глазами; в model.fit эти 2 координаты обычно не тащим.")


<a id="cmp"></a>
## 11. Сравнение (как на плакате)

### Главное различие

| Метод | Использует **y**? | Главная цель |
|-------|-------------------|--------------|
| **PCA** | Нет | Сохранить максимум **дисперсии / информации** |
| **LDA** | **Да** | Максимально **разделить классы** |
| **NCA** | **Да** | Сделать соседей **одного класса ближе** (под KNN) |
| **t-SNE** | Нет* | **Красиво визуализировать** |

\* Метки можно раскрасить на графике, но алгоритм t-SNE их не учит.

### Картинка-интуиция

```text
PCA:  оси туда, где облако данных длиннее
LDA:  оси туда, где классы лучше отделимы
NCA:  «подкрутить» метрику под соседей с теми же метками
t-SNE: разложить точки на плоскости, сохраняя соседство
```

### Типичный выбор

| Задача | Метод |
|--------|--------|
| Сжать признаки для logreg/SVM/KNN | **PCA** (+ scaler, Pipeline) |
| Сжать **с учётом классов** | **LDA** |
| Улучшить KNN | **NCA** + KNN |
| Нарисовать отчёт / EDA | **t-SNE** или PCA 2D |
| Деревья / бустинг | часто **ничего** из этого не нужно |


<a id="итог"></a>
## 12. Шпаргалка и исправления к типичным ошибкам

### Чеклист PCA

1. `StandardScaler` → `PCA` → модель  
2. `fit` только на train / внутри CV  
3. `n_components` — число **или** доля дисперсии (0.95)  
4. Смотреть `explained_variance_ratio_`, но решать по **CV-метрике**  
5. Не ждать interpretable «возрастов» от PC1  

### Частые неточности (и правда)

| Неточно | Верно |
|---------|--------|
| «PCA всегда улучшает модель» | Нет, он **не** смотрит на y |
| «t-SNE = замена PCA в пайплайне» | t-SNE для **визуализации** |
| «LDA = PCA с другим названием» | LDA **с учителем**, цель — классы |
| «После PCA fit на всём X» | Утечка; только train |
| «Деревьям обязателен PCA» | Обычно **нет** |
| VIF/Selection vs Extraction | Selection **выбирает**, Extraction **строит оси** |

### Мини-глоссарий

| | |
|--|--|
| `components_` | направления осей PCA |
| `explained_variance_ratio_` | доля дисперсии по осям |
| $S_b$, $S_w$ | меж- / внутриклассовая матрицы LDA |
| perplexity | параметр «внимания к соседям» в t-SNE |


### Мини-практика

1. На `load_breast_cancer` сравните CV accuracy: без PCA / PCA(0.95) / PCA(5).  
2. Нарисуйте cumulative explained variance.  
3. Сравните 2D PCA и 2D t-SNE (раскраска по классу).  
4. Обучите LDA как классификатор в Pipeline.  
5. Объясните одной фразой: почему PCA fit на train+test — leakage.


In [ ]:
# ===== Сводка практики =====
results = {}
for name, est in [
    ("no_pca", Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=2000))])),
    ("pca_95", Pipeline([("sc", StandardScaler()), ("pca", PCA(0.95)), ("m", LogisticRegression(max_iter=2000))])),
    ("pca_5", Pipeline([("sc", StandardScaler()), ("pca", PCA(5)), ("m", LogisticRegression(max_iter=2000))])),
    ("lda", Pipeline([("sc", StandardScaler()), ("m", LinearDiscriminantAnalysis())])),
]:
    results[name] = cross_val_score(est, X, y, cv=5, scoring="accuracy").mean()

print("CV accuracy:")
for k, v in results.items():
    print(f"  {k:8s}  {v:.3f}")
print("\nВывод: PCA не всегда лучше baseline; LDA здесь часто силён (классы в X разделимы).")


## Что делать дальше

1. Отбор признаков (**Selection**) и извлечение (**Extraction**) — разные инструменты; часто идут цепочкой: clean → select → PCA → model.  
2. Для продакшена: только методы с честным `transform` (PCA, LDA, NCA).  
3. t-SNE — в ноутбук отчёта, не в `predict` сервиса.

### Главная мысль

> **PCA** жмёт данные, сохраняя разброс.  
> **LDA/NCA** жмут с оглядкой на **классы**.  
> **t-SNE** рисует красивую карту.  
> Всегда: scale (где нужно) + fit на train + проверка метрикой, а не только «% дисперсии».

Удачи!
